In [1]:
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException,ElementNotInteractableException, NoSuchElementException

#from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(10) # 隱性等待
driver.get("https://today.line.me/tw/v2/movie/incinemas/playing")

# 初始化 ActionChains
action = ActionChains(driver)


In [3]:
# 初始化前一次抓取的電影數量
last_count = 0 

# 持續捲動頁面，直到沒有新資料載入為止
while True: 
    # 1. 取得目前頁面上所有的電影元素
    movie_list_whole = driver.find_elements(By.CSS_SELECTOR, 'div.detailListItem.movieListing-movie')
    current_count = len(movie_list_whole)

    # 2. 判斷是否還有新資料載入
    if current_count == last_count:
        print("數量未增加，判斷已抵達底部")
        break
    
    # 3. 找出目前清單中的「最後一個元素」
    last_element = movie_list_whole[-1]

    # 4. 使用 ActionChains 滾動到最後一個元素
    # 這會讓瀏覽器將視窗滾動到該元素出現的位置，通常會觸發Lazy Loading（延遲載入）
    action.scroll_to_element(last_element).perform()
    
    # 5. 等待新資料仔入 (時間依網速調整)
    time.sleep(2)

    # 6. 更新 last_count
    last_count = current_count

# 初始化一個空的列表，用來儲存所有抓取到的電影資訊
movie_list = []

# 遍歷所有電影元素並提取詳細資訊
for i, element in enumerate(movie_list_whole):
    # 取得電影標題
    title = element.find_element(By.CSS_SELECTOR, 'h2.detailListItem-title.header.header--sm.header--primary.header--ellipsis1').text

    # 取得內容分級
    # 初始化變數，避免後續引用錯誤
    content_rating = "無分級資訊"
    try:
        # 嘗試尋找分級元素
        rating_element = element.find_element(By.CSS_SELECTOR,'span.glnBadge-text.text.text--fNarrow.text--secondary.text--regular')
        content_rating = rating_element.text
    except NoSuchElementException:
        # 若找不到元素，程式不會崩潰，而是執行此區塊
        # 可以選擇 pass 或紀錄 log ，這裡維持預設值 " 無分級資訊 "
        pass

    showtimes_element = element.find_element(By.CSS_SELECTOR,'a.detailListItem-bookingButton')
    Showtimes_link = showtimes_element.get_attribute('href')

    
    print(f"{i+1}. {title} , 分級: {content_rating}, 場次連結: {Showtimes_link}")

    movie_list.append({
        "title": title,
        "Content_Rating": content_rating,
        "Showtimes_link": Showtimes_link
    })


數量未增加，判斷已抵達底部
1. 阿凡達：火與燼 , 分級: 保護級, 場次連結: https://today.line.me/tw/v3/movie/x2PRNDq/1
2. 動物方城市2 , 分級: 普遍級, 場次連結: https://today.line.me/tw/v3/movie/XYwXwN0/1
3. 冠軍之路 , 分級: 普遍級, 場次連結: https://today.line.me/tw/v3/movie/KwzZVxR/1
4. 陽光女子合唱團 , 分級: 輔12級, 場次連結: https://today.line.me/tw/v3/movie/1Dnwjm3/1
5. 家弒服務 , 分級: 輔15級, 場次連結: https://today.line.me/tw/v3/movie/5yeQgrR/1
6. 海綿寶寶電影版：尋寶大冒險 , 分級: 普遍級, 場次連結: https://today.line.me/tw/v3/movie/60lo0Z1/1
7. 大蟒蛇 , 分級: 輔12級, 場次連結: https://today.line.me/tw/v3/movie/3NwRYRZ/1
8. 叫我驅魔男神 , 分級: 輔15級, 場次連結: https://today.line.me/tw/v3/movie/kENK8oL/1
9. 大濛 , 分級: 保護級, 場次連結: https://today.line.me/tw/v3/movie/9m7LoZm/1
10. 那張照片裡的我們 , 分級: 輔12級, 場次連結: https://today.line.me/tw/v3/movie/peP5YO7/1
11. 顫懼 , 分級: 輔15級, 場次連結: https://today.line.me/tw/v3/movie/Ya5QqaL/1
12. 大衛：王者降臨 , 分級: 普遍級, 場次連結: https://today.line.me/tw/v3/movie/Za5y38L/1
13. 橫衝直闖 , 分級: 輔15級, 場次連結: https://today.line.me/tw/v3/movie/MLnM8Bw/1
14. 琳達！琳達！(4K數位修復版) , 分級: 普遍級, 場次連結: https://today.line.m

In [ ]:
# 初始化一個主列表來存放所有資料
all_reviews_data = []
FULL_STAR_PATH = "M12 18.344l-5.81 3.609c-.147.091-.34.046-.43-.1-.043-.07-.057-.152-.04-.23l1.469-6.96-5.09-4.736c-.126-.118-.133-.316-.016-.442.052-.056.122-.091.198-.099l6.746-.68 2.684-6.513c.066-.16.249-.235.408-.17.077.032.138.094.17.17l2.684 6.514 6.746.68c.172.017.297.17.28.342-.008.075-.043.146-.099.198l-5.09 4.736 1.47 6.96c.036.169-.072.334-.24.37-.08.017-.161.002-.23-.04L12 18.343z"
HALF_STAR_PATH = "M12.12 2.024c.076.031.137.093.169.17l2.684 6.513 6.746.68c.172.017.297.17.28.342-.008.075-.043.146-.099.198l-5.09 4.736 1.47 6.96c.036.169-.072.334-.24.37-.08.017-.161.002-.23-.04L12 18.343l-5.81 3.61c-.147.091-.34.046-.43-.1-.043-.07-.057-.152-.04-.23l1.469-6.96-5.09-4.736c-.126-.118-.133-.316-.016-.442.052-.056.122-.091.198-.099l6.746-.68 2.684-6.513c.066-.16.249-.235.408-.17zM12 6.463v9.651l3.662 2.275-.925-4.383 3.316-3.086-4.398-.443L12 6.463z"

# 遍歷電影列表中的每一部電影，抓取個別電影的評論資訊
for element in movie_list:
    # 取得電影名稱
    movie_name = element["title"]

    # 取得電影的場次連結
    individual_showtimes_link = element["Showtimes_link"]

    print(f"正在爬取電影: {movie_name}的評論...")
    # 前往該電影的詳細頁面
    driver.get(individual_showtimes_link)
    time.sleep(2)

    try:
        # 這邊要攻略「等待最多10秒，直到「評論」分頁元素變得可以點擊」
        # 教學目標 : 設定並且練習使用顯示等待 (Explicit Wait)的策略
        # 目標 : 建立一個等待物件，設定最常等待時間(10秒)。後續所有等待操作都須遵循這個超時設定
        wait = WebDriverWait(driver, 10)
        
        #教學目標: 練習將CSS選擇器字串存放在一個變數中，方便後續維護與修改
        review_tab_selector = 'a.ltcp-link.tabLink.movie.css-1hq811z'
        # 等待直到所有符合選擇器的分頁連結元素出現在頁面中
        elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, review_tab_selector)))
        if len(elements) < 3:
            print(f"電影: {movie_name} 的分頁數量不足，跳過...")
            continue
        '''           
        這邊要攻略「等待……直到「至少一個」符合選擇器的元素出現在DOM中，並取得所有符合的元素。」
        「如果你問老師說:那如果10秒內都沒有等到呢?」
        老師會回答你:「那就會拋出TimeoutException異常，執行將跳轉到對應的except區塊)
        '''
        #因為評論就是這個選擇器所對應的最後一個元素
        review_tab_to_click = elements[-1]
        #等待直到該元素可以被點擊，並執行點擊
        wait.until(EC.element_to_be_clickable(review_tab_to_click)).click()
    
        time.sleep(2)
    except (TimeoutException, IndexError, ElementNotInteractableException) as e:
        print(f"電影: {movie_name} 的評論分頁無法互動，原因: {type(e).__name__}，跳過...")
        continue

    except IndexError:
        print(f"電影: {movie_name} 沒有評論分頁，跳過...")
        continue
    
    #Last count for each movie's reviews
    last_count = 0

    while True:
        # 1. 取得目前頁面上所有的評論元素
        comment_list_whole = driver.find_elements(By.CSS_SELECTOR, 'div.css-hnvcda')
        current_count = len(comment_list_whole)

        # 2. 終止條件 : 判斷是否還有新資料載入
        # 如果目前數量等於上次數量，代表網頁已不再載入新內容，迴圈結束
        if current_count == last_count:
            break

        # 3. 鎖定目標 : 找出目前清單中的「最後一個」評論元素
        # Python 列表使用 index -1 來選取最後一項
        last_element = comment_list_whole[-1]

        # 4. 使用ActionChains 滾動到最後一個元素
        # 這會讓瀏覽器將視窗滾動到該元素出現的位置，模擬人類閱讀到底部的行為
        action.scroll_to_element(last_element).perform()

        # 5. 等待載入 : 給予伺服器回應與瀏覽器渲染的時間
        time.sleep(2)

        # 6. 更新狀態 : 將目前的數量記錄下來，供下一輪比對
        last_count = current_count
    
    # 遍歷當前頁面的所有評論
        for html in comment_list_whole:
            # 將該評論元素的 HTML 原始碼取出，並使用 BeautifulSoup 進行解析
            # 練習使用 .get_attribute()方法 : 可以抓取innerHTML
            # 也就是請瀏覽器把這個元素『裡面』的所有 HTML 原始碼吐出來給我，變成一個純字串。」
            soup = bs(html.get_attribute('innerHTML'), 'html.parser')
            # 進行評論的資料擷取，會不會有些電影還剛上映所以沒有評論? 加入錯誤處理以增強穩定性
            try:
                # 擷取評論者的名稱
                reviewer = soup.select_one('div.ratingCommentItemUser-name.css-1gt5j0z').text
                #選出所有星星評分區塊內的 <path>元素
                star_paths = soup.select('div.ratingStars path')
                star = 0
                for path in star_paths:
                    path_data = path['d'] #取得 path 的 d 屬性值
                    if path_data == FULL_STAR_PATH:
                        star += 1
                    elif path_data == HALF_STAR_PATH:
                        star += 0.5
                
                # 擷取評論的內容文字
                comment = soup.select_one('span.css-xcrf5c').text
                
                # 將擷取到的評論資料整理成字典，並存入總清單中
                all_reviews_data.append({
                    'movie_name': movie_name,
                    'reviewer': reviewer,
                    'star': star,
                    'comment': comment
                })
            except AttributeError:
                print(f"電影: {movie_name} 的某則評論資料不完整，跳過該評論...")
                continue

driver.get_full_page_screenshot_as_png()

# 步驟5:所有電影爬取完畢，使用Pandas呈現
print("\n所有電影評論爬取完畢，正在生成 DataFrame...")
df = pd.DataFrame(all_reviews_data)

# 顯示 DataFrame 的前幾筆資料和整體資訊
print("DataFrame Head:")
print(df.head())
print("\nDataFrame Info:")
df.info()

正在爬取電影: 阿凡達：火與燼的評論...
正在爬取電影: 動物方城市2的評論...
正在爬取電影: 冠軍之路的評論...
正在爬取電影: 陽光女子合唱團的評論...
正在爬取電影: 家弒服務的評論...
正在爬取電影: 海綿寶寶電影版：尋寶大冒險的評論...
正在爬取電影: 大蟒蛇的評論...
正在爬取電影: 叫我驅魔男神的評論...
正在爬取電影: 大濛的評論...
正在爬取電影: 那張照片裡的我們的評論...
正在爬取電影: 顫懼的評論...
正在爬取電影: 大衛：王者降臨的評論...
正在爬取電影: 橫衝直闖的評論...
正在爬取電影: 琳達！琳達！(4K數位修復版)的評論...
正在爬取電影: 佐賀偶像是傳奇 夢幻銀河樂園的評論...
正在爬取電影: 東京教父(4K數位修復版)的評論...
正在爬取電影: 冰雪壞皇后的評論...
正在爬取電影: 東京計程車的評論...
正在爬取電影: 空之境界劇場版：第三章 痛覺殘留的評論...
正在爬取電影: 劇場版「鬼滅之刃」無限城篇 第一章 猗窩座再襲的評論...
正在爬取電影: 格陵蘭雪女的評論...
正在爬取電影: 空之境界劇場版：第二章 殺人考察(前)的評論...
正在爬取電影: 血降的評論...
正在爬取電影: 創：戰神的評論...
正在爬取電影: 國寶的評論...
正在爬取電影: 劇場版 BanG Dream! It's MyGO!!!!! 後篇：唱吧、成為我們羈絆的詩歌&FILM LIVE的評論...
正在爬取電影: 進行曲的評論...
正在爬取電影: 一戰再戰的評論...
正在爬取電影: 魔法壞女巫：第二部的評論...
正在爬取電影: 破壞之王的評論...
正在爬取電影: 地母的評論...
正在爬取電影: 喇嘛與仁波切的評論...
正在爬取電影: 8號出口的評論...
正在爬取電影: 沒好婚姻的評論...
正在爬取電影: 甕靈的評論...
正在爬取電影: 劇場版 BanG Dream! It's MyGO!!!!! 前篇：春暖向陽，迷星之貓的評論...
正在爬取電影: 角頭－鬥陣欸的評論...
正在爬取電影: 黑白線人的評論...
正在爬取電影: 劇場版『鏈鋸人 蕾潔篇』的評論...
正在爬取電影: 出神入化3的評論...
正在爬取電影: 換乘真愛的評論...
正在爬取電影: 時間暫停的季節的評論...

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=59895): Read timed out. (read timeout=120)

In [5]:
print("\n所有電影評論爬取完畢，正在生成 DataFrame...")
df = pd.DataFrame(all_reviews_data)

# 顯示 DataFrame 的前幾筆資料和整體資訊
print("DataFrame Head:")
print(df.head())
print("\nDataFrame Info:")
df.info()


所有電影評論爬取完畢，正在生成 DataFrame...
DataFrame Head:
  movie_name          reviewer  star  \
0    阿凡達：火與燼             藤原 翔太   5.0   
1    阿凡達：火與燼           YUCHENG   5.0   
2    阿凡達：火與燼                阿文   5.0   
3    阿凡達：火與燼            自由付出代價   5.0   
4    阿凡達：火與燼  蕭羽良-SmoothERmine   5.0   

                                             comment  
0  有史以來最好看的一集\n情感豐富堆疊\n戰鬥畫面精彩刺激\n隨著劇情推進猶如踏入阿凡達\n最...  
1                           也太好看了吧！必須去戲院看才能體會很好的視覺感受  
2               畫面太讚。看到沒上廁所。飲料沒喝。太扯。了。要刷到爆炸。怎麼可以那麼好看  
3  超好看！只有詹姆斯能超越詹姆斯了，可以說是把故事講的很細又長且還不會拖\n讓我覺得4刷5刷都...  
4  🎬 《阿凡達：火與燼》觀後心得 🔥🌋\n下班後硬撐 3 個半小時 進戲院，還得面對隔天一早上...  

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 859 entries, 0 to 858
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   movie_name  859 non-null    object 
 1   reviewer    859 non-null    object 
 2   star        859 non-null    float64
 3   comment     859 non-null    object 
dtypes: float64(

In [ ]:
print(df)
#存中文字符之csv
df.to_csv('line_movie_reviews.csv', index=False, encoding='utf-8-sig')

In [ ]:
df

In [ ]:
driver.quit()